In [5]:
# Cell 1 - Load mesh and measure original height

import trimesh  # import trimesh for mesh processing
import numpy as np  # import numpy for numeric operations

# set the path to the mesh file
mesh_path = "third.glb"

# set the real body height in the same unit you want to use for scaling
# example: if the real height is 170 cm, set real_height = 170.0
real_height = 163.5

print("Loading mesh from:", mesh_path)

# load the mesh from file
loaded = trimesh.load(mesh_path)

# check whether the loaded object is a scene or a mesh
if isinstance(loaded, trimesh.Scene):
    print("Loaded object is a Scene. Combining all geometries into one mesh.")
    # combine all geometries in the scene into a single mesh
    mesh = trimesh.util.concatenate(tuple(loaded.geometry.values()))
else:
    print("Loaded object is a Mesh.")
    # use the loaded mesh directly
    mesh = loaded

print("Mesh loaded successfully.")
print("Number of vertices:", len(mesh.vertices))
print("Number of faces:", len(mesh.faces))

# get all vertex coordinates
vertices = mesh.vertices

# get the minimum y value assuming vertical axis is Y
min_y = vertices[:, 1].min()

# get the maximum y value assuming vertical axis is Y
max_y = vertices[:, 1].max()

# compute mesh height from lowest point to highest point
mesh_height = max_y - min_y

print("Lowest vertex (feet) Y:", min_y)
print("Highest vertex (head) Y:", max_y)
print("Measured mesh height:", mesh_height)

# compute the scaling factor
scale = real_height / mesh_height

print("Real height:", real_height)
print("Scaling factor:", scale)

Loading mesh from: third.glb
Loaded object is a Scene. Combining all geometries into one mesh.
Mesh loaded successfully.
Number of vertices: 18439
Number of faces: 36874
Lowest vertex (feet) Y: -1.7439051866531372
Highest vertex (head) Y: -0.011845882050693035
Measured mesh height: 1.7320593046024442
Real height: 163.5
Scaling factor: 94.3963059264462


In [6]:
# Cell 2 - Make the mesh watertight (safe + version-compatible)

print("Starting watertight repair process...")

# make a copy so the original mesh remains unchanged
watertight_mesh = mesh.copy()

# print initial watertight status
print("Initial watertight status:", watertight_mesh.is_watertight)

# only attempt repairs if NOT watertight
if not watertight_mesh.is_watertight:
    print("Mesh is NOT watertight. Attempting repairs...")

    # fill holes in the mesh
    watertight_mesh.fill_holes()
    print("Attempted to fill holes.")

    # fix normals to ensure consistency
    watertight_mesh.rezero()  # shift mesh closer to origin
    watertight_mesh.remove_infinite_values()  # remove invalid values
    watertight_mesh.process(validate=True)  # auto cleanup
    print("Processed mesh with trimesh.process().")

else:
    print("Mesh is already watertight. Skipping heavy repair steps.")

# final watertight check
print("Final watertight status:", watertight_mesh.is_watertight)

# save the mesh
repaired_mesh_path = "front_watertight.glb"
watertight_mesh.export(repaired_mesh_path)

print("Repaired mesh exported to:", repaired_mesh_path)

Starting watertight repair process...
Initial watertight status: True
Mesh is already watertight. Skipping heavy repair steps.
Final watertight status: True
Repaired mesh exported to: front_watertight.glb


In [7]:
# Cell 3 - Calculate body volume and scaled body volume

print("Calculating mesh volume...")

# use the repaired mesh for volume calculation
body_volume = watertight_mesh.volume

print("Raw mesh volume:", body_volume)

# calculate scaled body volume using scale^3
scaled_body_volume = body_volume * (scale ** 3)

print("Scaling factor cubed:", scale ** 3)
print("Scaled body volume:", scaled_body_volume)

Calculating mesh volume...
Raw mesh volume: 0.07765828434201338
Scaling factor cubed: 841133.6302066345
Scaled body volume: 65320.994624216764


In [8]:
# Cell 4 - Convert scaled body volume to weight

print("Converting scaled body volume to estimated weight...")

# compute weight using the given formula
weight = (scaled_body_volume * 1.01) / 1000

print("Estimated weight:", weight)
print("Done.")

Converting scaled body volume to estimated weight...
Estimated weight: 65.97420457045892
Done.
